<a href="https://colab.research.google.com/github/prasertrak/data-engineer/blob/main/The_ETL_Pipeline_Handbook_How_to_Build_a_Production_Grade_Pipeline_in_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!python --version

In [ ]:
!pandas --version

In [ ]:
!ipykernel --version

In [ ]:
!pip show ipykernel

In [ ]:
pip install requests pandas numpy ipykernel

In [ ]:

# Standard library imports
import json
import os
import random
import subprocess
from dataclasses import dataclass, field
from datetime import date, timedelta
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple

# Third-party libraries
import numpy as np
import pandas as pd
import requests


# Display config for notebooks
pd.set_option('display.max_columns', None)  # All columns will show
# Prevents cutting long column text with ...
pd.set_option('display.max_colwidth', None)

print("All imports successful")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
import ipykernel
print(ipykernel.__version__)

In [ ]:
@dataclass
class APIConfig:
    """API configuration for HubEau data fetching.

    Think of this as the "address book" for the API.
    """
    use_mock: bool = True
    base_url: str = "https://hubeau.eaufrance.fr/api/v2/hydrometrie/obs_elab"
    metric: str = "HIXnJ"  # Daily max water level (elaborated observations)
    # Pagination: fetch 20k records per request (API Limit)
    max_per_page: int = 20000
    timeout_seconds: int = 60  # Network timeout

    def __post_init__(self):
        """Validate configuration after initialization."""
        if self.max_per_page <= 0:
            raise ValueError("max_per_page must be positive")
        if self.timeout_seconds <= 0:
            raise ValueError("timeout_seconds must be positive")


@dataclass
class StationConfig:
    """Station monitoring configuration.

    The 'what' of data collection: which stations, what's a flood?

    P.-S. There are five monitoring stations, three are historic and two are contemporary.
    """
    codes: List[str] = field(default_factory=lambda: [
        "F700000109",
        "F700000110",
        "F700000111",
        "F700000102",
        "F700000103",
    ])
    flood_threshold_mm: int = 6000  # Flood alert threshold
    earliest_date: str = "1900-01-01"  # How far back to go


@dataclass
class KaggleConfig:
    """Kaggle dataset publishing configuration.

    The 'destination' config: where to publish, what metadata to attach.

    #TODO: this can be optimized.
    """
    dataset_slug: str = "grimespoint/paris-flood-dataset"
    input_csv: str = "/kaggle/input/datasets/{slug}/paris_flood_dataset.csv"
    output_dir: Path = field(default_factory=lambda: Path(
        "kaggle/working/kaggle_dataset"))
    mock_output_dir: Path = field(default_factory=lambda: Path("mock_output"))
    output_filename: str = "paris_flood_dataset.csv"
    mock_output_filename: str = "mock_flood_dataset.csv"
    metadata_filename: str = "dataset-metadata.json"

    # Metadata
    title: str = "Paris flood dataset"
    keywords: list = field(default_factory=lambda: [
        "tabular", "weather and climate", "environment", "europe", "time series analysis"
    ])
    geospatial_coverage: str = "Paris, France"
    update_frequency: str = "Weekly"
    license_name: str = "CC0-1.0"

    # Computed fields (set in __post_init__)
    output_csv_path: Path = field(init=False)
    metadata_path: Path = field(init=False)

    def __post_init__(self):
        """Compute derived paths after initialization."""
        """ Derived attributes: compute once here, use everywhere"""
        self.input_csv = self.input_csv.format(slug=self.dataset_slug)
        self.output_csv_path = self.output_dir / self.output_filename
        self.metadata_path = self.output_dir / self.metadata_filename
        self.mock_output_filename = self.mock_output_dir / self.mock_output_filename


# Initialize configs
API_CONFIG = APIConfig()
STATION_CONFIG = StationConfig()
KAGGLE_CONFIG = KaggleConfig()

print("Configurations initialized")
print(f"  - Monitoring {len(STATION_CONFIG.codes)} stations")
print(f"  - Flood alert at: {STATION_CONFIG.flood_threshold_mm}mm")
print(f"  - Publishing to: kaggle.com/{KAGGLE_CONFIG.dataset_slug}")

In [ ]:
print("=== API_CONFIG ===")
print(API_CONFIG)
print()
print("=== STATION_CONFIG ===")
print(STATION_CONFIG)
print()
print("=== KAGGLE_CONFIG (key paths) ===")
print(f"  input_csv       : {KAGGLE_CONFIG.input_csv}")
print(f"  output_csv_path : {KAGGLE_CONFIG.output_csv_path}")
print(f"  metadata_path   : {KAGGLE_CONFIG.metadata_path}")

In [ ]:
def load_csv(path: str) -> pd.DataFrame:
    """Load CSV file or return empty DataFrame if file does not exist.

    Args:
        path (str): Full path to the CSV file.

    Returns:
        pd.DataFrame: Loaded data, or empty DataFrame if file not found.

    Raises:
        pd.errors.ParserError: If the CSV is malformed.
    """
    if os.path.exists(path):
        return pd.read_csv(path, low_memory=False, parse_dates= True, delimiter=",")
    return pd.DataFrame()   # Null Object: consistent return type

In [ ]:
# Test : load from a path that does not exist → empty DataFrame, no crash
df_missing = load_csv("/tmp/does_not_exist.csv")
print(f"Type    : {type(df_missing)}")
print(f"Shape   : {df_missing.shape}")
print(f"Empty?  : {df_missing.empty}")

# This means callers can always do df.empty instead of 'if df is None'
if df_missing.empty:
    print("No existing data. Will run a full fetch from earliest date.")

In [ ]:
def determine_update_range(existing: pd.DataFrame) -> Tuple[bool, Optional[str]]:
    """Determine whether an update is needed and from what date.

    Logic:
    1. Check if existing data covers yesterday's date
    2. If yes → no update needed
    3. If no → start fetching from day after last data

    Why only update to yesterday?
    - Today's data may not be finalized yet
    - Conservative approach (avoid incomplete data)

    Args:
        existing: Existing dataset (any schema)

    Returns:
        Tuple[should_update, start_date]
        - should_update: False if current, True if update needed
        - start_date: ISO date string to fetch from, None if no update
    """
    # Case 1
    if existing.empty:
        print("No existing data found. Will fetch all data from earliest date.")
        return True, STATION_CONFIG.earliest_date

    if "date_obs_elab" in existing.columns:
        record_colname = "date_obs_elab"
    elif "record_date" in existing.columns:
        record_colname = "record_date"
    else:
        raise KeyError(
            "Missing date column: expected 'date_obs_elab' or 'record_date'")

    s = pd.to_datetime(existing[record_colname], errors="coerce")
    last_day = s.max().date()

    yesterday = date.today() - timedelta(days=1)

    print(f"Last day in dataset: {last_day}")
    print(f"Yesterday: {yesterday}")

    # Case 2
    if last_day >= yesterday:
        print("\nDataset already covers yesterday or later. No update needed.")
        return False, None

    # Case 3
    next_day = (last_day + pd.Timedelta(days=1))
    print(f"\nWill retrieve data starting from: {next_day}")

    return True, next_day.isoformat()

In [ ]:
# Test 1: No existing data
print("=== Test 1: No existing data ===")
should_update, start_date = determine_update_range(pd.DataFrame())
print(f"Should update: {should_update}, Start from: {start_date}\n")

# Test 2: Old existing data
print("=== Test 2: Old existing data (Jan 10-15) ===")
old_data = pd.DataFrame({
    "date_obs_elab": pd.date_range("2026-01-10", periods=6),
    "code_station": ["F700000109"] * 6,
    "resultat_obs_elab": [5800.0] * 6,
})
should_update, start_date = determine_update_range(old_data)
print(f"Should update: {should_update}, Start from: {start_date}\n")

# Test 3: Recent existing data (covers yesterday)
print("=== Test 3: Recent data (covers yesterday) ===")
yesterday = date.today() - timedelta(days=1)
recent_data = pd.DataFrame({
    "date_obs_elab": pd.date_range(yesterday - timedelta(days=5), yesterday),
    "code_station": ["F700000109"] * 6,
    "resultat_obs_elab": [5800.0] * 6,
})
should_update, start_date = determine_update_range(recent_data)
print(f"Should update: {should_update}, Start from: {start_date}")

In [ ]:
response = requests.get(
    "https://hubeau.eaufrance.fr/api/v2/hydrometrie/obs_elab",
    params={
        "code_entite": "F700000109",
        "grandeur_hydro_elab": "HIXnJ",
        "date_debut_obs_elab": "1900-01-01",
        "size": 20000,
    },
    timeout=60,
)

data = response.json()
print("จำนวนข้อมูลทั้งหมด:", data.get("count"))


In [ ]:
requests.get(
    "https://hubeau.eaufrance.fr/api/v2/hydrometrie/obs_elab",
    params={
        "code_entite": "F700000109",
        "grandeur_hydro_elab": "HIXnJ",
        "date_debut_obs_elab": "1900-01-01",
        "size": 20000,
    }
)

In [ ]:
def generate_mock_api_data(station_code: str, start_date: str, num_days: int = 10) -> List[Dict]:
    """Generate realistic mock API data for demonstration.

    Simulates what HubEau API would return: list of observation dicts.

    Args:
        station_code: Station ID (e.g., 'F700000109')
        start_date: Start date (ISO format)
        num_days: How many days of data to generate

    Returns:
        List of observation dictionaries (API format)
    """

    start = pd.to_datetime(start_date).date()
    records = []

    validation_statuses = ["Donnée validée",
                           "Donnée brute", "Donnée pré-validée"]
    qualities = ["Bonne", "Non qualifiée", "Douteuse"]
    methods = ["Mesurée", "Calculée", "Expertisée"]

    for i in range(num_days):
        obs_date = start + timedelta(days=i)

        # Simulate realistic water levels (with some variation)
        base_level = 5500 + int(station_code[-2:])  # Varies by station
        noise = random.randint(-200, 200)
        water_level = base_level + noise

        record = {
            "code_site": "mock_" + station_code[1:],
            "code_station": "mock_" + station_code,
            "date_obs_elab": obs_date.isoformat(),
            "resultat_obs_elab": water_level,
            "date_prod": (obs_date + timedelta(days=1)).isoformat(),
            "code_statut": "1",
            "libelle_statut": random.choice(validation_statuses),
            "code_methode": "1",
            "libelle_methode": random.choice(methods),
            "code_qualification": "1",
            "libelle_qualification": random.choice(qualities),
            "longitude": 2.3522 + random.uniform(-0.01, 0.01),
            "latitude": 48.8566 + random.uniform(-0.01, 0.01),
            "grandeur_hydro_elab": "mock_HIXnJ",
        }
        records.append(record)

    return records

In [ ]:
# Generate mock data
mock_api_response = generate_mock_api_data(
    "F700000109", "2026-01-15", num_days=5)

print(f"Generated {len(mock_api_response)} mock API records")
print(f"\nSample record:")

print(json.dumps(mock_api_response[0], indent=2))

In [ ]:
def fetch_single_station_data(station_code: str, start_date: str, use_mock: bool = True) -> pd.DataFrame:
    """Fetch all hydrometric data for a single station from mock API (default) or from the real endpoint.

    In production (cursor based pagination strategy):
    - Fetches max_per_page records per request
    - Continues until no new data or yesterday's date reached - If the last row's date < yesterday → advance cursor → fetch next page
    - Stop when: no data returned | last date >= yesterday | page was not full
    - Handles network errors gracefully

    NOTE : uses mock data for safety and simplicity.

    Args:
        station_code: station code (e.g., 'F700000109')
        start_date: start date for retrieval (ISO format)
        use_mock: If True, use mock data instead of real API

    Returns:
        DataFrame with raw API data (not yet processed)
    """
    if use_mock:
        # Simulate API response
        data = generate_mock_api_data(station_code, start_date, num_days=7)
        page_df = pd.DataFrame(data)
        # Parse to dt here for convenience
        page_df["date_obs_elab"] = pd.to_datetime(
            page_df["date_obs_elab"], errors="coerce").dt.normalize()
        return page_df

    # Real implementation
    else:
        session = requests.Session()  # Reuse TCP connection across pages
    frames = []
    cursor = start_date

    while True:
        params = {
            "code_entite": station_code,
            "grandeur_hydro_elab": API_CONFIG.metric,
            "date_debut_obs_elab": cursor,
            "size": API_CONFIG.max_per_page,
        }

        try:
            response = session.get(
                API_CONFIG.base_url,
                params=params,
                timeout=API_CONFIG.timeout_seconds  # Best practice, always set
            )
            response.raise_for_status()
        except requests.RequestException as e:  # Don't let one station kill the whole pipeline

            print(f"Error fetching data for station {station_code}: {e}")
            break

        data = response.json().get("data", [])
        if not data:  # Empty response : we've exhausted this station
            break

        page_df = pd.DataFrame(data)
        page_df["date_obs_elab"] = pd.to_datetime(
            page_df["date_obs_elab"], errors="coerce").dt.normalize()
        frames.append(page_df)

        last_page_date = page_df["date_obs_elab"].max()
        yesterday = date.today() - timedelta(days=1)

        # Prevent infinite loops
        if pd.isna(last_page_date) or last_page_date.date() >= yesterday:
            break

        cursor = (last_page_date + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

        if len(data) < API_CONFIG.max_per_page:
            break

    if frames:
        return pd.concat(frames, ignore_index=True)
    return pd.DataFrame()

In [ ]:
def fetch_all_data(start_date: str, use_mock: bool = True) -> pd.DataFrame:
    """Orchestrator: Fetch data for all configured stations.

    This is the high-level "Extract" step in our ETL pipeline.

    Args:
        start_date: start date for retrieval
        use_mock: use mock data or real API

    Returns:
        merged_historical_and_new DataFrame from all stations
    """
    frames = []

    # For demo, use only first 2 stations
    # stations_to_fetch = STATION_CONFIG.codes[:2]

    for station_code in STATION_CONFIG.codes:
        print(f"Fetching data for station {station_code}...")
        df_station = fetch_single_station_data(
            station_code, start_date, use_mock=use_mock)

        if not df_station.empty:
            print(f"  Got {len(df_station)} records")
            frames.append(df_station)
        else:
            print(f"  (no data)")

    if frames:
        return pd.concat(frames, ignore_index=True)
    return pd.DataFrame()

In [ ]:
# Fetch fresh data
raw_data = fetch_all_data("2026-01-15", use_mock=True)

print(f"\nTotal raw records fetched: {len(raw_data)}")
print(f"\nFirst few rows (API schema):")
print(raw_data[["code_station", "date_obs_elab", "resultat_obs_elab"]].head())

# Transform

In [ ]:
def convert_to_date(df: pd.DataFrame, columns: List[str]) -> pd.DataFrame:
    """Convert specified columns to pandas datetime type.

    Handles:
    - Valid ISO dates: "2026-01-15" → datetime
    - Bad dates: "not a date" → NaT (Not a Time, pandas' null for datetimes)
    - Missing values: NaN → NaT

    Args:
        df: Input DataFrame
        columns: Column names to process

    Returns:
        New DataFrame with parsed and validated datetime columns
    """
    df = df.copy()  # Never modify the original!

    for col in columns:
        if col in df.columns:
            # errors='coerce' converts unparseable values to NaT
            df[col] = pd.to_datetime(df[col], errors="coerce").dt.normalize()

    return df


def convert_to_numeric(df: pd.DataFrame, columns: List[str]) -> pd.DataFrame:
    """Convert specified columns to numeric (float) type.

    Handles:
    - Valid numbers: 5800, "5800.5" → float
    - Bad numbers: "N/A", "unknown" → NaN
    - Missing values: NaN → NaN (unchanged)

    Args:
        df: Input DataFrame
        columns: Column names to process

    Returns:
        New DataFrame with parsed and validated numeric columns
    """
    df = df.copy()

    for col in columns:
        if col in df.columns:
            # errors='coerce' converts unparseable values to NaN
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

In [ ]:
messy_df = pd.DataFrame({
    "date_obs_elab": ["2026-01-15", "2026-01-16", "not a date", None],
    "resultat_obs_elab": [5800.0, "5900", "N/A", None],
})

print("BEFORE type conversion:")
print(messy_df)
print(f"\nTypes: {messy_df.dtypes.to_dict()}")

type_safe_df = convert_to_date(messy_df, ["date_obs_elab"])
type_safe_df = convert_to_numeric(type_safe_df, ["resultat_obs_elab"])

print("\nAFTER type conversion:")
print(type_safe_df)
print(f"\nTypes: {type_safe_df.dtypes.to_dict()}")

print("\nBad values coerced to NaT/NaN (pipeline continues safely)")

In [ ]:
def auto_convert_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Auto-detect and convert datetime and numeric columns to the correct type.

    Analyzes column content to determine if columns should be parsed as
    datetimes or numerics, then applies appropriate type conversion.

    Args:
        df: Input DataFrame

    Returns:
        DataFrame with converted columns
    """
    datetime_cols = []
    numeric_cols = []

    for col in df.columns:
        # Skip columns that are already properly typed
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            continue

        # For object/string columns, infer the intended type
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):

            # Get a non-null sample to test
            sample = df[col].dropna().head(10)

            if len(sample) == 0:
                continue

            # Try datetime parsing first
            try:
                pd.to_datetime(sample, errors='raise', format='mixed')
                datetime_cols.append(col)
                continue
            except (ValueError, TypeError):
                pass

            # Try numeric parsing
            try:
                pd.to_numeric(sample, errors='raise')
                numeric_cols.append(col)
                continue
            except (ValueError, TypeError):
                pass

    # Apply conversion functions with detected columns
    df = convert_to_date(df, datetime_cols)
    df = convert_to_numeric(df, numeric_cols)

    return df

In [ ]:
# ============================================================================
# SCHEMA MAPPINGS
# ============================================================================

# Primary mapping: French API columns to English column names
API_TO_EN = {
    "code_site": "location_code",
    "code_station": "station_code",
    "date_obs_elab": "record_date",
    "resultat_obs_elab": "water_level_mm",
    "date_prod": "data_production_date",
    "code_statut": "validation_status_code",
    "libelle_statut": "validation_status",
    "code_methode": "production_method_code",
    "libelle_methode": "production_method",
    "code_qualification": "quality_code",
    "libelle_qualification": "quality_assessment",
    "longitude": "longitude",
    "latitude": "latitude",
    "grandeur_hydro_elab": "hubeau_elab_code",
}

# Reverse mapping: English to French (computed automatically)
EN_TO_API = {v: k for k, v in API_TO_EN.items()}

print("API → English mapping sample:")
for api_col, en_col in list(API_TO_EN.items())[:5]:
    print(f"  {api_col:25} {en_col}")

print(f"\n{len(API_TO_EN)} column mappings defined")

In [ ]:
# ============================================================================
# CATEGORICAL VALUE MAPPINGS
# ============================================================================

# Translation tables for categorical columns
CATEGORICAL_MAPPINGS = {
    "validation_status": {
        "Donnée validée": "validated",
        "Donnée brute": "raw",
        "Donnée pré-validée": "pre-validated",
    },
    "quality_assessment": {
        "Bonne": "good",
        "Non qualifiée": "unqualified",
        "Douteuse": "dubious",
    },
    "production_method": {
        "Calculée": "calculated",
        "Mesurée": "measured",
        "Expertisée": "expert-reviewed",
    },
}

print("Categorical mappings (French → English):")
for col_name, mapping in CATEGORICAL_MAPPINGS.items():
    print(f"\n  {col_name}:")
    for fr, en in mapping.items():
        print(f"    '{fr}' : '{en}'")

In [ ]:
def rename_to_api_schema(df: pd.DataFrame) -> pd.DataFrame:
    """Rename English column names back to API schema names.

    Why? Defensive programming. If we need to re-export to API format.

    Best practice: use .copy() to avoid modifying original DataFrame
    (prevents unexpected side effects in calling code)

    Args:
        df: DataFrame with English column names

    Returns:
        New DataFrame with API column names (original unchanged)
    """
    if df.empty:
        return df.copy()

    # Only rename columns that exist (defensive)
    columns_to_rename = {k: v for k, v in EN_TO_API.items() if k in df.columns}
    return df.rename(columns=columns_to_rename)


def rename_to_english(df: pd.DataFrame) -> pd.DataFrame:
    """Rename API column names to English schema names.

    This is the main transformation we use during processing.
    Only rename when the target English column is not already present.

    Args:
        df: DataFrame with API column names

    Returns:
        New DataFrame with English column names (original unchanged)
    """

    if df.empty:
        return df.copy()

    columns_to_rename = {}
    for src, dst in API_TO_EN.items():
        if src in df.columns and dst not in df.columns:
            columns_to_rename[src] = dst

    return df.rename(columns=columns_to_rename)

In [ ]:
# preserve any values that don't match.
def apply_categorical_mappings(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()
    for col_name, mapping in CATEGORICAL_MAPPINGS.items():
        if col_name in df.columns:
            df[col_name] = df[col_name].map(mapping).fillna(df[col_name])
    return df

In [ ]:
# Test the functions
sample_api_df = pd.DataFrame({
    "code_station": ["F700000109"],
    "date_obs_elab": ["2024-01-15"],
    "resultat_obs_elab": [5800.0],
    "libelle_statut": ["Donnée validée"],
})

print("Original (API Schema):")
print(sample_api_df.columns.tolist())

renamed_df = rename_to_english(sample_api_df)
print("\nAfter rename_to_english():")
print(renamed_df.columns.tolist())

reversed_df = rename_to_api_schema(renamed_df)
print("\nAfter rename_to_api_schema() [should match original]:")
print(reversed_df.columns.tolist())

print("\nRound-trip rename successful (bidirectional mapping works).")

In [ ]:
def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Add computed columns based on raw data.

    Currently adds:
    - flood_alert: Boolean flag if water_level_mm > flood_threshold_mm

    Why add derived columns?
    - Helps downstream analysis (filtering, alerting)
    - Pre-computed in ETL = faster queries
    - Logic centralized in pipeline

    Args:
        df: DataFrame with English column names

    Returns:
        New DataFrame with derived columns
    """
    df = df.copy()

    if "water_level_mm" in df.columns:
        df["flood_alert"] = df["water_level_mm"] > STATION_CONFIG.flood_threshold_mm

    return df

In [ ]:
# ============================================================================
# COLUMN ORDERING
# ============================================================================

# User defined preferred column order (logical grouping)
COLUMN_ORDER = [
    # Primary identifiers & measurements
    "station_code",
    "record_date",
    "water_level_mm",
    "flood_alert",  # Derived metric (computed)

    # Metadata about the observation
    "hubeau_elab_code",
    "data_production_date",
    "validation_status_code",
    "validation_status",
    "production_method_code",
    "production_method",
    "quality_code",
    "quality_assessment",

    # Geographic info (less important)
    "location_code",
    "longitude",
    "latitude",
]

print(f"Column order defined: {len(COLUMN_ORDER)} columns")
print(f"Priority: IDs & measurements → metadata → geography")

In [ ]:
def order_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Reorder columns to preferred order.

    Why reorder?
    - Logical grouping (IDs, measurements, metadata, geography)
    - User experience (most important columns first)
    - Consistency across runs

    Args:
        df: DataFrame with any column order

    Returns:
        DataFrame with reordered columns
    """
    # Columns in preferred order that exist
    present_cols = [c for c in COLUMN_ORDER if c in df.columns]
    # Remaining columns (if any)
    other_cols = [c for c in df.columns if c not in present_cols]

    return df[present_cols + other_cols]

In [ ]:
def create_dedup_key(df: pd.DataFrame) -> pd.Series:
    """Create unique deduplication key from station, day, and water level value.

    Key format: "station_code_YYYY-MM-DD_value"
    Example: "F700000109_2024-01-15_5800.0"

    Why this format?
    - Readable (human-inspectable)
    - Hashable (can use in sets for fast lookup)
    - Immutable components (no time-of-day, just date)

    Args:
        df: DataFrame

    Returns:
        Series of deduplication keys (NaN for rows with missing components)
    """
    parts = []

    # Station code
    if "code_station" in df.columns:
        parts.append(df["code_station"].astype(str))

    # Date (day only, no time)
    if "date_obs_elab" in df.columns:
        parts.append(df["date_obs_elab"].dt.strftime("%Y-%m-%d"))

    # Water level value
    if "resultat_obs_elab" in df.columns:
        parts.append(df["resultat_obs_elab"].astype(str))

    if not parts:
        return pd.Series(index=df.index, dtype="object")

    # Combine parts with underscore separator
    return pd.Series(
        ["_".join(row) for row in zip(*parts)],
        index=df.index
    )

In [ ]:
# Test
test_std_df = pd.DataFrame({
    "code_station": ["F700000109", "F700000110"],
    "date_obs_elab": pd.to_datetime(["2026-01-15", "2026-01-16"]),
    "resultat_obs_elab": [5800.0, 5900.0],
})

keys = create_dedup_key(test_std_df)
print("Deduplication Keys:")
print(keys.tolist())

print("\nKeys are unique, human-readable, and immutable")

In [ ]:
def remove_duplicates(existing: pd.DataFrame, new: pd.DataFrame) -> pd.DataFrame:
    """Remove rows from 'new' that already exist in 'existing'.

    Algorithm:
    1. Enforce types in both DataFrames
    2. Create dedup keys for both
    3. Find keys in 'existing'
    4. Filter 'new' to exclude those keys
    5. Return only the truly new rows

    Args:
        existing: existing dataset (any schema)
        new: new data to filter (same schema as existing)

    Returns:
        Copy of 'new' with duplicate rows removed
    """
    # Short-circuit: if either is empty, no work to do
    if existing.empty or new.empty:
        return new.copy()

    # Parse types to both for comparison
    existing_std = auto_convert_columns(existing)
    new_std = auto_convert_columns(new)

    # Create dedup keys
    existing_keys = set(create_dedup_key(existing_std).dropna())
    new_keys = create_dedup_key(new_std)

    # Boolean mask: True where the new row is genuinely new
    mask = ~new_keys.isin(existing_keys)

    # Return only new rows (preserve original schema)
    # Index back into original (non-standardised) 'new' to preserve all columns
    result = new.iloc[new_keys[mask].index].copy()

    return result

In [ ]:
# Test with duplicate scenario
existing_test_data = pd.DataFrame({
    "code_station": ["F700000109", "F700000109"],
    "date_obs_elab": ["2026-01-15", "2026-01-16"],
    "resultat_obs_elab": [5800.0, 5850.0],
})

new_test_data = pd.DataFrame({
    "code_station": ["F700000109", "F700000109", "F700000110"],
    # First row is duplicate
    "date_obs_elab": ["2026-01-15", "2026-01-17", "2026-01-17"],
    "resultat_obs_elab": [5800.0, 5900.0, 6100.0],
})

print("Existing data (2 rows):")
print(existing_test_data)

print("\nNew data (3 rows, 1 is duplicate):")
print(new_test_data)

deduped = remove_duplicates(existing_test_data, new_test_data)

print("\nAfter deduplication (2 rows):")
print(deduped)

print(f"\nRemoved {len(new_test_data) - len(deduped)} duplicate row(s), will append these two rows to the existing dataset.")

In [ ]:
def postprocess(df: pd.DataFrame) -> pd.DataFrame:
    """Apply all post-processing transformations.

    The complete "Transform" pipeline:
    1. Enforce datetime and numeric types
    2. Rename to English schema
    3. Translate categorical values
    4. Add derived columns
    5. Reorder columns logically
    6. Sort by date and station & reset index

    Args:
        df: Raw API data (French schema)

    Returns:
        Fully processed, publication-ready dataset
    """
    if df.empty:
        return df

    df = df.copy()

    # Step 1: Type conversions
    print("  1. Converting types...")
    df = auto_convert_columns(df)

    # Step 2: Rename columns
    print("  2. Renaming columns (French → English)...")
    df = rename_to_english(df)

    # Step 3: Map categories
    print("  3. Mapping categorical values...")
    df = apply_categorical_mappings(df)

    # Step 4: Add computed columns
    print("  4. Adding derived columns...")
    df = add_derived_columns(df)

    # Step 5: Organize columns
    print("  5. Reordering columns...")
    df = order_columns(df)

    # Step 6: Sort and reset index
    print("  6. Sorting and resetting index...")
    df = df.sort_values(["record_date", "station_code"]).reset_index(drop=True)

    return df

In [ ]:
# Test the full postprocessing pipeline
print("Processing raw API data...")
raw_data_fr = rename_to_english(raw_data)
raw_data_fr = apply_categorical_mappings(raw_data_fr)

processed_mock_data = postprocess(raw_data_fr)

print(f"\nProcessing complete: {len(processed_mock_data)} rows")
print(f"\nFirst row (processed, English schema):")
print(processed_mock_data.iloc[0])

print(f"\nColumn order verified: ")
print(processed_mock_data.columns.tolist())

# THE LOAD STEP

In [ ]:
def create_output_dir(use_mock: bool = False) -> None:
    """Create output directory (and parents) if it does not already exist.

    Args:
        use_mock (bool): If True, use mock_output directory instead of configured output_dir.
    """
    output_dir = KAGGLE_CONFIG.mock_output_dir if use_mock else KAGGLE_CONFIG.output_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    # exist_ok=True: idempotent : safe to call multiple times

In [ ]:
create_output_dir(use_mock=False)

In [ ]:
def create_metadata(df: pd.DataFrame, config: KaggleConfig) -> Dict:
    """Generate Kaggle dataset metadata from DataFrame and config.

    - Dataset discoverability (keywords, title)
    - Usage rights (license)
    - Data freshness (temporal coverage)
    - Geographic context

    Args:
        df: Processed DataFrame with 'record_date' column
        config: KaggleConfig instance

    Returns:
        Dictionary in Kaggle-compatible format
    """
    if df.empty or "record_date" not in df.columns:
        first_date = "unknown"
        last_date = "unknown"
    else:
        first_date = df["record_date"].min().strftime("%Y-%m-%d")
        last_date = df["record_date"].max().strftime("%Y-%m-%d")

    metadata = {
        "title": config.title,
        "id": config.dataset_slug,
        "licenses": [{"name": config.license_name}],
        "keywords": config.keywords,
        "temporalCoverage": {
            "startDate": first_date,
            "endDate": last_date,
        },
        "geospatialCoverage": config.geospatial_coverage,
        "updateFrequency": config.update_frequency,
    }

    return metadata

# Generate metadata
create_output_dir(use_mock=True)
metadata = create_metadata(processed_mock_data, KAGGLE_CONFIG)

print("Dataset Metadata:")
print(json.dumps(metadata, indent=2))

# Save metadata to JSON
metadata_path = KAGGLE_CONFIG.mock_output_dir
dataset_meta_file = metadata_path / KAGGLE_CONFIG.metadata_filename

with open(dataset_meta_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"\nMetadata saved to {metadata_path}")

In [ ]:
print(mock_api_response)

In [ ]:
def run_etl_pipeline(use_mock: bool = True) -> pd.DataFrame:
    """Run the complete ETL pipeline: Extract → Transform → Load to CSV.

    Returns:
        Processed DataFrame ready for publication
    """
    print("\n" + "="*70)
    print("PARIS FLOOD DATASET ETL PIPELINE")
    print("="*70)

    # ====================================================================
    # STEP 1: LOAD EXISTING DATA
    # ====================================================================

    if use_mock:
        print("\n=== mock pipeline ===")
        print("\n[STEP 1] Loading mock dataset...")
        # For demo, use a sample existing dataset
        loaded_df = pd.DataFrame(mock_api_response)
        loaded_df['date_obs_elab'] = pd.to_datetime(loaded_df['date_obs_elab'])

        # Demo : translate in english here (mimic loaded CSV)
        loaded_df = rename_to_english(loaded_df)
        print(f"  Loaded {len(loaded_df)} existing records")

    else:
        print("\n=== API pipeline ===")
        print("\n[STEP 1] Loading existing dataset...")
        loaded_df = load_csv(KAGGLE_CONFIG.input_csv)
        print(f"  {len(loaded_df):,} existing rows loaded.")

    # ====================================================================
    # STEP 2: DETERMINE UPDATE RANGE
    # ====================================================================
    print("\n[STEP 2] Determining update range...")
    should_update, start_date = determine_update_range(loaded_df)
    if not should_update:
        print("\n Dataset is current. Returning existing data.")
        return loaded_df

    # ====================================================================
    # STEP 3: EXTRACT (FETCH DATA)
    # ====================================================================
    print(f"\n[STEP 3] Extracting data from {start_date})...")
    fetched_data = fetch_all_data(start_date, use_mock=use_mock)
    print(f"  Extracted {len(fetched_data)} raw records")

    # ====================================================================
    # STEP 4: TRANSFORM I : DEDUPLICATE
    # ====================================================================
    print("\n[STEP 4] Deduplicating against existing data...")
    deduped_fetched_data = remove_duplicates(loaded_df, fetched_data)
    print(f"  Removed {len(fetched_data) - len(deduped_fetched_data)} duplicates")
    print(f"  {len(deduped_fetched_data)} truly new records to add")

    # ====================================================================
    # STEP 5: TRANSFORM II : COMBINE (MERGE WITH EXISTING)
    # ====================================================================
    print("\n[STEP 5] Merging new data with existing...")
    # Translate fetched data into English (match loaded_df) before combining
    new_df_english = rename_to_english(deduped_fetched_data)
    merged_historical_and_new = pd.concat([loaded_df, new_df_english], ignore_index=True)
    print(f"  Combined dataset: {len(merged_historical_and_new)} total records")

    # ====================================================================
    # STEP 6: TRANSFORM III : POST-PROCESS
    # ====================================================================
    print("\n[STEP 6] Postprocessing data...")
    processed_records = postprocess(merged_historical_and_new)
    print(f"  Transformation complete")

    # ====================================================================
    # STEP 7: EXPORT (LOAD TO CSV)
    # Mock exports only mock CSV
    # For real world behaviour, set mock=False in the later section
    # ====================================================================
    print("\n[STEP 7] Exporting to CSV...")

    create_output_dir(use_mock=use_mock)

    if use_mock:
        output_path = KAGGLE_CONFIG.mock_output_filename
    else:
        output_path = KAGGLE_CONFIG.output_csv_path

    processed_records.to_csv(output_path, index=False, sep=",")

    file_size_kb = os.path.getsize(output_path) / 1024
    print(f"  Exported to {output_path}")
    print(f"  File size: {file_size_kb:.1f} KB")

    print("\n" + "="*70)
    print("ETL PIPELINE COMPLETE")
    print("="*70)

    return processed_records


# Run the pipeline
mock_dataset = run_etl_pipeline(use_mock=True)

In [ ]:
def write_metadata(df: pd.DataFrame) -> None:
    """Write dataset metadata to JSON file.

    Calls create_metadata to generate content and writes to path specified
    in KAGGLE_CONFIG.metadata_path.

    Args:
        df (pd.DataFrame): Processed DataFrame to extract temporal coverage from.

    Raises:
        IOError: If file writing fails.
    """
    metadata = create_metadata(df, KAGGLE_CONFIG)

    with open(KAGGLE_CONFIG.metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f"Metadata written to {KAGGLE_CONFIG.metadata_path}")

In [ ]:
def publish_to_kaggle() -> None:
    """Publish the updated dataset to Kaggle using the Kaggle CLI.

    Runs 'kaggle datasets version' command with current output directory.
    Includes timestamp in the version message.

    Raises:
        subprocess.CalledProcessError: If Kaggle CLI command fails.
    """
    timestamp = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
    message = f"Weekly update: {timestamp}"

    cmd = [
        "kaggle",
        "datasets",
        "version",
        "-p",
        str(KAGGLE_CONFIG.output_dir),
        "-m",
        message,
        "--dir-mode",
        "zip",
    ]

    print("Publishing to Kaggle...")
    print("Command:", " ".join(cmd))

    try:
        subprocess.run(cmd, check=True)
        print("Successfully published to Kaggle.")
    except subprocess.CalledProcessError as e:
        print(f"Error publishing to Kaggle: {e}")
        raise

In [ ]:
def validate_and_analyze(df: pd.DataFrame) -> None:
    """Perform post-pipeline validation and analysis.

    Checks:
    - No unexpected null values
    - Correct data types
    - Data range sanity
    - Flood alerts computed
    """
    print("\n" + "="*40)
    print("DATA VALIDATION & ANALYSIS")
    print("="*40)

    print(f"\nDataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")

    print("\nData Types:")
    for col, dtype in df.dtypes.items():
        print(f"  {col:30} {str(dtype):15}")

    print("\nNull Values:")
    null_counts = df.isnull().sum()
    null_cols = null_counts[null_counts > 0]
    if len(null_cols) == 0:
        print("  No null values found")
    else:
        for col, count in null_cols.items():
            pct = (count / len(df)) * 100
            print(f"  {col:30} {count:5} ({pct:5.1f}%)")

    print("\nWater Level Statistics (mm):")
    water_level_stats = df["water_level_mm"].describe()
    for stat_name, value in water_level_stats.items():
        if pd.notna(value):
            print(f"  {stat_name:10} {value:10.1f}")

    print(
        f"\nFlood Alerts (threshold: {STATION_CONFIG.flood_threshold_mm}mm):")
    flood_count = df["flood_alert"].sum()
    flood_pct = (flood_count / len(df)) * 100 if len(df) > 0 else 0
    print(f"  {flood_count} records ({flood_pct:.1f}%) exceed threshold")

    print(f"\nTemporal Coverage:")
    print(f"  From: {df['record_date'].min()}")
    print(f"  To:   {df['record_date'].max()}")
    days_span = (df['record_date'].max() - df['record_date'].min()).days
    print(f"  Span: {days_span} days")

    print(f"\nStations Covered:")
    stations = df["station_code"].unique()
    for i, station in enumerate(sorted(stations), 1):
        count = (df["station_code"] == station).sum()
        print(f"  {i}. {station}: {count} records")

    print(f"\nData Quality Summary:")
    print(
        f"  Validation Status Categories: {df['validation_status'].nunique()}")
    print(
        f"  Quality Assessment Categories: {df['quality_assessment'].nunique()}")

    print("\n" + "="*70)

In [ ]:
def main() -> None:
    """Execute the full Paris Flood Monitoring ETL pipeline.

    Flow:

    EXTRACT
    1. Load the existing dataset from the Kaggle input mount
    2. Determine whether an update is needed (and from what date)
    3. Fetch new data from the Hub'Eau API

    TRANSFORM
    4. Deduplicate against the existing dataset
    5. Combine and post-process (type parsing, translation, derived cols...)

    LOAD
    6. Write the updated CSV and metadata file
    7. Publish the new dataset version to Kaggle

    Exits early (code 0) if the dataset is already up to date.
    """
    print("=" * 70)
    print("Paris Flood Monitoring : ETL Pipeline (with real API calls)")
    print("=" * 70)

    final_dataset = run_etl_pipeline(use_mock=False)
    final_dataset.to_csv(KAGGLE_CONFIG.output_csv_path, index=False)
    print(f"      CSV written to {KAGGLE_CONFIG.output_csv_path}")
    write_metadata(final_dataset)

    print("\n[FINAL STEP] Publishing to Kaggle...")
    publish_to_kaggle()

    print("\n API Pipeline complete.")

    print("Running post-run validation")
    validate_and_analyze(final_dataset)

In [ ]:
if __name__ == "__main__":
    main()